# سبق 02 - مائیکروسافٹ ایجنٹ فریم ورک کی تلاش

**مائیکروسافٹ ایجنٹ فریم ورک (MAF)** ایک متحدہ فریم ورک ہے جو AI ایجنٹس بنانے کے لیے تیار کیا گیا ہے۔ یہ ایک صاف، مرکب کردہ آرکیٹیکچر فراہم کرتا ہے جس کے چار بنیادی تعمیراتی بلاکس ہیں:

- **کلائنٹ** – AI ماڈل اینڈ پوائنٹ سے جڑتا ہے اور رابطے کو سنبھالتا ہے
- **ایجنٹ** – کلائنٹ کو ہدایات اور ٹول کی تعریفات کے ساتھ لپیٹتا ہے
- **ٹولز** – ماڈل کی کال کر سکنے والی کسٹم فنکشنز کے ذریعے ایجنٹ کی صلاحیتوں کو بڑھاتے ہیں
- **سیشن** – کثیر گردش بات چیت کے لیے گفتگو کی تاریخ کو برقرار رکھتا ہے

اس سبق میں، ہم ایک **سفر کی بکنگ ایجنٹ** بنائیں گے جو ان تصورات کو استعمال کرتے ہوئے منزل کی دستیابی چیک کرتا ہے۔


## سیٹ اپ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ایجنٹ فریم ورک آرکیٹیکچر کو سمجھنا

مائیکروسافٹ ایجنٹ فریم ورک ایک تہہ دار آرکیٹیکچر کی پیروی کرتا ہے:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **کلائنٹ** – ایک `AzureAIProjectAgentProvider` Azure OpenAI ڈیپلائمنٹ سے جڑتا ہے۔ یہ تصدیق، درخواست کی فارمیٹنگ، اور جواب کی تشریح کو سنبھالتا ہے۔
2. **ایجنٹ** – کلائنٹ سے `provider.create_agent()` کے ذریعے بنایا جاتا ہے، ایجنٹ ماڈل تک رسائی کو ہدایات (سسٹم پرامپٹ) اور آلات کے ساتھ جوڑتا ہے۔
3. **آلات** – پائتھن فنکشنز جن پر `@tool` لگایا گیا ہو، جنہیں ایجنٹ کارروائی کرنے یا ڈیٹا حاصل کرنے کے لیے کال کر سکتا ہے۔
4. **سیشن** – ایک `AgentSession` آبجیکٹ (جو `agent.create_session()` کے ذریعے بنایا جاتا ہے) جو گفتگو کی تاریخ کو محفوظ کرتا ہے، جس سے کثیر مرحلہ وار مکالمہ ممکن ہوتا ہے جہاں ایجنٹ پچھلے سیاق و سباق کو یاد رکھتا ہے۔

آئیں ہر تہہ کو قدم بہ قدم تیار کریں۔


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool سجاوٹی کے ساتھ ٹولز شامل کرنا

ٹولز ایجنٹس کو متن تیار کرنے سے آگے کارروائیاں کرنے دیتے ہیں۔ `@tool` سجاوٹی ایک معمول کے Python فنکشن کو ایسی چیز میں تبدیل کر دیتی ہے جسے ایجنٹ کال کر سکتا ہے۔

اہم نکات:
- `Annotated[type, "description"]` استعمال کریں تاکہ ماڈل ہر پیرامیٹر کو سمجھ سکے۔
- ڈاک اسٹرنگ وہ ٹول کی وضاحت بن جاتی ہے جو ماڈل دیکھتا ہے۔
- `approval_mode="never_require"` کا مطلب ہے کہ ٹول خودکار طور پر بغیر صارف کی تصدیق کے چلتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ایجنٹ کو ٹولز کے ساتھ بنانا

اب ہم کلائنٹ، ہدایات، اور ٹولز کو ایک ایجنٹ میں جوڑتے ہیں۔ `instructions` نظام کے پرامپٹ کے طور پر کام کرتی ہیں — یہ ایجنٹ کے کردار اور رویے کی تعریف کرتی ہیں۔


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## سیشن کے ساتھ کثیر مرلہ بات چیت

ایک `AgentSession` (جو `agent.create_session()` کے ذریعے بنایا جاتا ہے) گفتگو میں تمام پیغامات کا حساب رکھتا ہے۔ ہر `agent.run()` کال میں ایک ہی سیشن پاس کرنے سے، ایجنٹ کو پوری گفتگو کا مکمل ریکارڈ دستیاب ہوتا ہے اور وہ پہلے کے پیغامات کا حوالہ دے سکتا ہے۔

ہم `tools=[check_destination_availability]` پاس کرتے ہیں تاکہ ایجنٹ ہر مرلہ میں ہمارے دستیابی چیکر کو کال کر سکے۔


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## خلاصہ

اس سبق میں آپ نے مائیکروسافٹ ایجنٹ فریم ورک کے چار ستونوں کو دریافت کیا:

| تصور | آپ نے کیا سیکھا |
|---------|------------------|
| **کلائنٹ** | `AzureAIProjectAgentProvider` تصدیقی بنیاد پر Azure OpenAI سے رابطہ کرتا ہے |
| **ایجنٹ** | `provider.create_agent()` ایک ماڈل کنکشن کو ہدایات اور نام کے ساتھ جوڑتا ہے |
| **اوزار** | `@tool` ڈیکوریٹر ایجنٹ کے کال کرنے کے لیے پائتھن فنکشنز کو ظاہر کرتا ہے |
| **سیشن** | `agent.create_session()` متعدد موڑوں کے دوران گفتگو کا ریکارڈ رکھتا ہے |

یہ تعمیراتی بلاکس مل کر ایسے ایجنٹس بناتے ہیں جو قدرتی گفتگو کر سکتے ہیں، بیرونی فنکشنز کو کال کر سکتے ہیں، اور سیاق و سباق کو برقرار رکھ سکتے ہیں — جو بعد کے اسباق میں مزید ترقی یافتہ ایجنٹک پیٹرنز کی بنیاد ہے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دستیابی کی اعلامیہ**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کا استعمال کرتے ہوئے ترجمہ کی گئی ہے۔ جب کہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستی ہو سکتی ہے۔ اصل دستاویز اپنی مادری زبان میں قابلِ اعتماد ماخذ سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمے کی سفارش کی جاتی ہے۔ ہم اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کے ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
